# Paper §5.2 — K-subspace sweep (multi-direction within a layer)

Reproduces the §5.2 pilot grid (PlotQA × OneVision, L × α × K) and the 5-dataset layer sweep at L=26 (K=8 vs K=1 fallback). Both rest on an (a − m) calibration of the OneVision Main residual stream.

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §5.2 (K-subspace sweep).

This notebook drives the heavy inference stages by `subprocess`-invoking
the existing sharded drivers in `scripts/`. The `RUN_INFERENCE = False`
toggle below lets a reviewer read the entire pipeline without GPU
access. Full reproduction targets the **8 × H200** cluster and uses
`--gpus 0,1,2,3,4,5,6,7` end-to-end.


## 1 · Setup — paths, GPU sharding, subprocess helper

In [ ]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    # Gitignored artifacts (inputs/, outputs/, docs/insights/_data/) live in
    # the main worktree even when this notebook runs from a linked worktree.
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    # Current worktree's working tree top (== main when not in a linked worktree).
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Outputs live under MAIN (gitignored); figures land under WORKTREE so they ride the active branch.
# Scripts + configs come from WORKTREE so the active branch's edits are used
# (subprocess invocations would otherwise hit the stale main-worktree copies
# until the active PR merges).
SCRIPTS    = WORKTREE / "scripts"
CONFIGS    = WORKTREE / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"

# §5.2 reads predictions.jsonl. Under H1 (raw-number prompt + eps=0 DF)
# point at the paper2 snapshot. Legacy timestamped run dirs preserved as
# fallback for audit-trail / direct rerun-without-H1 reproduction.
H1_PRED_ROOT = MAIN / "outputs" / "paper2" / "cross_model_cross_dataset" / "predictions"
H1_PRED_ROOTS = {
    "plotqa":         H1_PRED_ROOT / "plotqa"         / "llava-onevision-qwen2-7b-ov",
    "infographicvqa": H1_PRED_ROOT / "infographicvqa" / "llava-onevision-qwen2-7b-ov",
    "chartqa":        H1_PRED_ROOT / "chartqa"        / "llava-onevision-qwen2-7b-ov",
    "mathvista":      H1_PRED_ROOT / "mathvista"      / "llava-onevision-qwen2-7b-ov",
    "tallyqa":        H1_PRED_ROOT / "tallyqa"        / "llava-onevision-qwen2-7b-ov",
}
LEGACY_PRED_ROOTS = {
    "plotqa":         MAIN / "outputs" / "experiment_e7_plotqa_full"         / "llava-onevision-qwen2-7b-ov" / "20260502-132624",
    "infographicvqa": MAIN / "outputs" / "experiment_e7_infographicvqa_full" / "llava-onevision-qwen2-7b-ov" / "20260502-152105",
    "chartqa":        MAIN / "outputs" / "experiment_e5e_chartqa_full"       / "llava-onevision-qwen2-7b-ov" / "20260502-211028",
    "mathvista":      MAIN / "outputs" / "experiment_e5e_mathvista_full"     / "llava-onevision-qwen2-7b-ov" / "20260502-212440",
    "tallyqa":        MAIN / "outputs" / "experiment_e5e_tallyqa_full"       / "llava-onevision-qwen2-7b-ov" / "20260502-083926",
}


def legacy_pred(ds_tag: str) -> Path:
    return LEGACY_PRED_ROOTS[ds_tag] / "predictions.jsonl"


def h1_pred(ds_tag: str) -> Path:
    return H1_PRED_ROOTS[ds_tag] / "predictions.jsonl"

# §5.2 e6_steering input root selection (same toggle as §5.1 above):
#   - RUN_INFERENCE=False: read pre-existing sweep dirs from legacy
#     `outputs/e6_steering/` (the aggregators have always filtered by
#     subdirectory name, so the legacy tree won't pollute results even
#     when read-only).
#   - RUN_INFERENCE=True: write new calibrate / sweep dirs to an isolated
#     tree so this run doesn't commingle with the legacy pool.
E6_ROOT_LEGACY = MAIN / "outputs" / "e6_steering"
E6_ROOT_FRESH  = MAIN / "outputs" / "paper2" / "section_5_e6_steering"

# §5.1 attention input root selection:
#   - RUN_INFERENCE=False: read pre-existing bbox-with runs from
#     legacy outputs/attention_analysis/ (the docstring header in
#     `extract_attention_mass.py` confirms each model has multiple
#     timestamped runs carrying `image_anchor_digit`; the analyzer
#     auto-skips bbox-less records via field-presence check).
#   - RUN_INFERENCE=True: write new runs to a fresh isolated tree so
#     they do not commingle with the legacy pool.
ATT_ROOT_LEGACY = MAIN / "outputs" / "attention_analysis"
# `section_5_attention` is the n=400 root (n=400 spec was the first run).
# `section_5_attention_n1000` is the n=1000 extension. After n=1000 completes,
# n=400 is to be retired and only the n=1000 tree kept.
ATT_ROOT_FRESH  = MAIN / "outputs" / "paper2" / "section_5_attention_n1000"
PEAKS_CSV       = MAIN / "outputs" / "paper2" / "section_5_attention_n1000" / "_data" / "cross_dataset_peaks.csv"
BBOX_FILE       = MAIN / "inputs" / "irrelevant_number_bboxes.json"

ATT_ROOT_FRESH.mkdir(parents=True, exist_ok=True)
PEAKS_CSV.parent.mkdir(parents=True, exist_ok=True)
assert BBOX_FILE.exists(), f"missing digit-pixel bbox JSON: {BBOX_FILE}"

PDF_OUT = MAIN     / "outputs" / "paper2" / "section_5_figures"
PNG_OUT = WORKTREE / "docs"    / "figures"
PDF_OUT.mkdir(parents=True, exist_ok=True)
PNG_OUT.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4")  # 5 GPUs by default
RUN_INFERENCE = os.environ.get("VLM_ANCHOR_RUN_INFERENCE", "").lower() in ("1", "true", "yes")  # set VLM_ANCHOR_RUN_INFERENCE=1 to invoke the heavy sharded drivers.

# Pick the attention + e6_steering input roots.
# `ATT_ROOT_FRESH` (section_5_attention_n1000) holds the canonical §5.1
# inference output, so always read from it — the RUN_INFERENCE toggle
# only gates whether a new launch script *adds* to that tree.
ATT_ROOT = ATT_ROOT_FRESH
E6_ROOT  = E6_ROOT_FRESH if RUN_INFERENCE else E6_ROOT_LEGACY
E6_ROOT_FRESH.mkdir(parents=True, exist_ok=True)

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


In [ ]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    # Print and (optionally) execute a shell command from the main worktree.
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str):
    pdf = PDF_OUT / f"{stem}.pdf"
    png = PNG_OUT / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {pdf}")
    print(f"wrote {png}")


## 2 · Configuration

OneVision Main is `llava-onevision-qwen2-7b-ov`. The calibration scope
follows outline §6.1 — PlotQA + InfoVQA pooled, wrong-base + numeric
`(a, m)` pairs only.

The pilot grid is 4 layers × 3 α × 4 K = 48 cells (PlotQA, n=250) —
K=1 included per outline §5.2 body figure (K=1/2/4/8 monotonic
improvement). The 5-dataset sweep adds 5 datasets × 4 layers at K=8 +
5 datasets × 4 layers at K=1, all at α=1.0 (40 cells total).


In [ ]:
ONEVISION    = "llava-onevision-qwen2-7b-ov"
ONEVISION_HF = "llava-hf/llava-onevision-qwen2-7b-ov-hf"

PILOT_LAYERS = [14, 16, 20, 26]  # H1 §5.1 peak: OneVision=14 (Main), qwen2.5-7b=16; 20/26 retained per user spec
PILOT_ALPHAS = [0.5, 1.0, 2.0]
PILOT_KS     = [1, 2, 4, 8]

CALIB_SCOPE       = "plotqa_infovqa_pooled_h1"
CALIB_MAX_PAIRS   = 2500

# Batched generate. B=16 validated via _smoke_batched_vs_sequential.py
# (parsed_number exact-match vs preserved sequential run on the same sids +
# cells). B=1 falls back to the legacy per-sample path. Prefetch workers
# parallelise PIL.Image.open across the batch and pipeline the next chunk
# while the current chunk runs on GPU.
SWEEP_BATCH_SIZE     = 1   # batched generate causes bf16 drift on OneVision AnyRes — see [[batched-generate-onevision-anyres]]
SWEEP_PREFETCH_WORKERS = 16

SWEEP_DATASETS_5D = [
    ("plotqa",    "experiment_e7_plotqa_full"),
    ("infovqa",   "experiment_e7_infographicvqa_full"),
    ("tallyqa",   "experiment_e5e_tallyqa_full"),
    ("chartqa",   "experiment_e5e_chartqa_full"),
    ("mathvista", "experiment_e5e_mathvista_full"),
]
SWEEP_LAYERS_5D = [14, 16, 20, 26]
SWEEP_KS_5D     = [8, 1]   # K=8 sweep + K=1 fallback at L=26
SWEEP_ALPHA_5D  = 1.0


## 3 · Seed the (a − m) subspace into the fresh E6 root

The §5.2 K-subspace sweep is the part being reproduced here. The
upstream calibration step — pooled SVD over PlotQA + InfoVQA
wrong-base + numeric (a, m) pairs (outline §6.1) — already produced
`outputs/e6_steering/<model>/_subspace/subspace_plotqa_infovqa_pooled_n5k_K16.pt`
(shape `[n_layers, K_max=16, d_model]`) on the canonical §4 predictions.
We symlink that audited subspace into the isolated `E6_ROOT_FRESH`
tree so the sweep below points at it without re-running calibration.

Re-running the pooled SVD would require `merge_calibrate_subspaces.py`,
which is not in-repo; the pooled subspace is treated here as a §4
derived artifact (analogous to `predictions.jsonl`).


In [ ]:
def seed_canonical_subspace():
    src = E6_ROOT_LEGACY / ONEVISION / "_subspace" / f"subspace_{CALIB_SCOPE}_K16.pt"
    dst_dir = E6_ROOT_FRESH / ONEVISION / "_subspace"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"subspace_{CALIB_SCOPE}.pt"
    if not src.exists():
        raise FileNotFoundError(
            f"missing canonical pooled subspace: {src}. "
            "Re-run E6 calibration on PlotQA + InfoVQA pool before §5.2."
        )
    if dst.is_symlink() or dst.exists():
        dst.unlink()
    dst.symlink_to(src)
    print(f"  symlink {dst} -> {src}")


seed_canonical_subspace()


## 4 · Pilot grid sweep — PlotQA × OneVision (5-GPU sharded)

`PILOT_LAYERS × PILOT_ALPHAS × PILOT_KS = 4 × 3 × 3 = 36 cells`. Each
cell shards its `(sid × cond)` inference across the 5 GPUs configured
via `VLM_ANCHOR_GPUS`; cells run sequentially. The chosen cell from
the §6.2.2 selection rule (`CHOSEN = (26, 8, 1.0)`) sits inside this
grid as the star marker after aggregation. Expected wall on 5 H200 ≈
1–2 GPU-hours total (≈ 100–200 s per cell × 36).


In [ ]:
PILOT_OUT_CSV = E6_ROOT_FRESH / "_data" / "E6_pilot_grid.csv"
PILOT_OUT_CSV.parent.mkdir(parents=True, exist_ok=True)


def sweep_pilot():
    subspace_path = E6_ROOT_FRESH / ONEVISION / "_subspace" / f"subspace_{CALIB_SCOPE}.pt"
    out_dir = E6_ROOT_FRESH / ONEVISION / "pilot_grid_plotqa_n250"
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "run_sweep_subspace_sharded.py"),
        "--config", str(CONFIGS / "experiment_e7_plotqa_full.yaml"),
        "--model", ONEVISION, "--hf-model", ONEVISION_HF,
        "--predictions-path", str(h1_pred("plotqa")),
        "--dataset-tag", "plotqa",
        "--subspace-path", str(subspace_path),
        "--subspace-scope", CALIB_SCOPE,
        "--sweep-layers", ",".join(str(L) for L in PILOT_LAYERS),
        "--sweep-alphas", ",".join(str(a) for a in PILOT_ALPHAS),
        "--sweep-ks",     ",".join(str(k) for k in PILOT_KS),
        "--max-samples", "250",
        "--batch-size", str(SWEEP_BATCH_SIZE),
        "--prefetch-workers", str(SWEEP_PREFETCH_WORKERS),
        "--gpus", GPUS,
        "--out-dir", str(out_dir),  # isolate to E6_ROOT_FRESH
    ]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


sweep_pilot()
run_cmd(
    ["uv", "run", "python", str(SCRIPTS / "aggregate_e6_pilot_grid.py"),
     "--e6-root", str(E6_ROOT_FRESH),
     "--layers", ",".join(str(L) for L in PILOT_LAYERS),
     "--ks",     ",".join(str(k) for k in PILOT_KS),
     "--alphas", ",".join(str(a) for a in PILOT_ALPHAS),
     "--out-csv", str(PILOT_OUT_CSV),
     "--fig-dir", str(MAIN / "outputs" / "paper2" / "section_5_figures")],
    dry=not RUN_INFERENCE,
)


## 5 · Figure §5.2a — pilot grid heatmap (4 metrics × K × L × α)

In [ ]:
def fig_pilot_grid() -> plt.Figure | None:
    src = PILOT_OUT_CSV
    if not src.exists():
        print(f"  (skipped — {src} missing; run §4 with RUN_INFERENCE=True first)")
        return None
    grid = pd.read_csv(src)
    grid = grid[grid["calib"] == "plotqa"]
    chosen_L, chosen_K, chosen_alpha = 26, 8, 1.0

    metrics = [
        ("delta_adopt", "Δ adopt(a) pp"),
        ("delta_df",    "Δ df(a) pp"),
        ("delta_em_a",  "Δ em(a) pp"),
        ("delta_em_b",  "Δ em(b) pp"),
    ]
    fig, axes = plt.subplots(len(metrics), len(PILOT_KS),
                             figsize=(11, 9.0), dpi=150,
                             sharex=True, sharey=True)
    for col, K in enumerate(PILOT_KS):
        for row, (col_metric, ylabel) in enumerate(metrics):
            ax = axes[row, col]
            sub = grid[grid["K"] == K]
            piv = sub.pivot_table(index="layer", columns="alpha", values=col_metric)
            piv = piv.reindex(index=PILOT_LAYERS, columns=PILOT_ALPHAS)
            piv = piv * 100.0  # to pp
            vmax = float(piv.abs().values.max()) if piv.size else 1.0
            ax.imshow(piv.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
            for i, L in enumerate(PILOT_LAYERS):
                for j, alpha in enumerate(PILOT_ALPHAS):
                    v = piv.loc[L, alpha]
                    if pd.notna(v):
                        ax.text(j, i, f"{v:+.1f}", ha="center", va="center",
                                fontsize=8, color="black")
                    if (L, K, alpha) == (chosen_L, chosen_K, chosen_alpha):
                        ax.add_patch(plt.Rectangle((j - 0.45, i - 0.45),
                                                   0.9, 0.9, fill=False,
                                                   edgecolor="black", linewidth=2.0))
            if row == 0:
                ax.set_title(f"K={K}", fontsize=11)
            if col == 0:
                ax.set_ylabel(ylabel, fontsize=10)
            ax.set_xticks(range(len(PILOT_ALPHAS))); ax.set_xticklabels(PILOT_ALPHAS, fontsize=8)
            ax.set_yticks(range(len(PILOT_LAYERS))); ax.set_yticklabels(PILOT_LAYERS, fontsize=8)
            if row == len(metrics) - 1:
                ax.set_xlabel("α", fontsize=9)
    fig.suptitle("§5.2a — E6 pilot grid (PlotQA × OneVision, n=250)\n"
                 "Δ vs baseline; ▢ chosen cell = L=26, K=8, α=1.0 (§6.2.2 selection)",
                 fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


fig = fig_pilot_grid()
if fig is not None:
    save_figure(fig, "paper_5_2a_E6_pilot_grid_plotqa")
fig


## 6 · 5-dataset layer sweep — K=8 main + K=1 fallback at L=26

Cross-dataset cross-check of the K choice. Each `(dataset, L)` cell is
one sharded sweep run; cells run sequentially. Expected ≈ 6–10 GPU-hours
on 8 × H200.


In [ ]:
SWEEP_OUT_DATA = E6_ROOT_FRESH / "_data"
SWEEP_OUT_DATA.mkdir(parents=True, exist_ok=True)


def sweep_5dataset_layer():
    subspace_path = E6_ROOT_FRESH / ONEVISION / "_subspace" / f"subspace_{CALIB_SCOPE}.pt"
    # Map §5.2 dataset_tag to the legacy_pred() key; tallyqa+chartqa+mathvista
    # share names. PlotQA / InfoVQA aliases reconcile the §5 short tag with the
    # extract-side full tag used by extract_attention_mass.py.
    pred_key = {"plotqa": "plotqa", "infovqa": "infographicvqa",
                "tallyqa": "tallyqa", "chartqa": "chartqa",
                "mathvista": "mathvista"}
    for ds_tag, cfg_slug in SWEEP_DATASETS_5D:
        pred = h1_pred(pred_key[ds_tag])
        out_dir = E6_ROOT_FRESH / ONEVISION / f"sweep_subspace_{ds_tag}_{CALIB_SCOPE}_p4_layer_sweep_K1_layers_K8"
        cmd = [
            "uv", "run", "python", str(SCRIPTS / "run_sweep_subspace_sharded.py"),
            "--config", str(CONFIGS / f"{cfg_slug}.yaml"),
            "--model", ONEVISION, "--hf-model", ONEVISION_HF,
            "--predictions-path", str(pred),
            "--dataset-tag", ds_tag,
            "--subspace-path", str(subspace_path),
            "--subspace-scope", CALIB_SCOPE,
            "--sweep-layers", ",".join(str(L) for L in SWEEP_LAYERS_5D),
            "--sweep-alphas", str(SWEEP_ALPHA_5D),
            "--sweep-ks", ",".join(str(k) for k in SWEEP_KS_5D),
            "--gpus", GPUS,
            "--out-dir", str(out_dir),  # isolate to E6_ROOT_FRESH
        ]
        run_cmd(cmd, dry=not RUN_INFERENCE)

    run_cmd(
        ["uv", "run", "python", str(SCRIPTS / "aggregate_e6_layer_sweep_p4.py"),
         "--e6-root", str(E6_ROOT_FRESH),
         "--out-data", str(SWEEP_OUT_DATA),
         "--out-fig", str(MAIN / "outputs" / "paper2" / "section_5_figures")],
        dry=not RUN_INFERENCE,
    )


sweep_5dataset_layer()


## 7 · Figure §5.2b — 5-dataset Δdf(a) sweep + K=1 fallback

In [ ]:
def fig_layer_sweep() -> plt.Figure | None:
    src = SWEEP_OUT_DATA / "p4_layer_sweep_per_cell_ci.csv"
    if not src.exists():
        print(f"  (skipped — {src} missing; run §6 with RUN_INFERENCE=True first)")
        return None
    sweep = pd.read_csv(src)
    # Canonical CSV uses `layer`, `delta_df`, and `ds_tag` for the dataset key.

    fig, ax = plt.subplots(figsize=(11, 5.2), dpi=150)
    color = {"plotqa": "#1F4FA8", "infovqa": "#C8102E",
             "tallyqa": "#1A7F3F", "chartqa": "#F2A900", "mathvista": "#6C7280"}
    for ds_tag, c in color.items():
        head = sweep[(sweep["ds_tag"] == ds_tag) & (sweep["K"] == 8)]
        if len(head):
            head = head.sort_values("layer")
            ax.plot(head["layer"], head["delta_df"] * 100,
                    color=c, marker="o", label=f"{ds_tag} K=8")
        k1 = sweep[(sweep["ds_tag"] == ds_tag) & (sweep["K"] == 1)]
        if len(k1):
            k1 = k1.sort_values("layer")
            ax.plot(k1["layer"], k1["delta_df"] * 100,
                    color=c, marker="s", linestyle="--", alpha=0.6,
                    label=f"{ds_tag} K=1")
    ax.axhline(0, color="#888", linewidth=0.7, linestyle=":")
    ax.set_xlabel("layer (L)")
    ax.set_ylabel("Δ df(a)  pp  (negative ⇒ anchoring reduced)")
    ax.set_title("§5.2b — 5-dataset Δdf(a) at α=1.0 — K=8 (solid) vs K=1 (dashed)\n"
                 "K=1 sign-reversal at mid-stack supports §5.2 K-subspace argument")
    ax.legend(loc="upper right", frameon=False, fontsize=8, ncol=2)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    fig.tight_layout()
    return fig


fig = fig_layer_sweep()
if fig is not None:
    save_figure(fig, "paper_5_2b_layer_sweep_delta_df")
fig


## Summary

Pipeline: calibrate (a−m) subspace on PlotQA+InfoVQA pooled → pilot
grid sweep on PlotQA × OneVision → 5-dataset layer sweep at K∈{8,1}
→ two figures (`paper_5_2a_E6_pilot_grid_plotqa`,
`paper_5_2b_layer_sweep_delta_df`).

Full reproduction on 8 × H200 ≈ 12–16 GPU-hours; calibration alone is
≈ 2 hours. The §5.3 narrative notebook synthesizes the §5.1 + §5.2
outputs.
